# 09 · Manuscript figures and tables

No training. Builds every figure and table in the IEEE Access manuscript from the result files of notebooks 01-08, and writes `manuscript/provenance.csv`. That file lists, for each figure and table, the result file it came from, the notebook and date of the run that made that file, and the library version.

Upload `manuscript/figures/*.pdf` and `manuscript/tables/*.tex` to the `figures/` and `tables/` folders in Overleaf. Figure 1 (the pipeline diagram) is drawn in LaTeX and is not made here.

The numbers written in the manuscript text are **not** updated automatically. After any rerun, compare the printed values below with the text.

| | |
|---|---|
| Trains models | no |
| Needs | notebooks 01-08 |
| Writes | `manuscript/figures/`, `manuscript/tables/`, `manuscript/provenance.csv`, `manuscript/rareguard_figures_tables.zip` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import json, zipfile, datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

FIG = rg.proj(cfg, "manuscript", "figures")
TAB = rg.proj(cfg, "manuscript", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
PROVENANCE = []

def run_info(exp):
    with open(rg.proj(cfg, "results", exp, "run_info.json")) as fh:
        return json.load(fh)

def record(item, out_file, sources):
    """sources: list of (experiment, file inside results/<experiment>/)"""
    for exp, name in sources:
        info = run_info(exp)
        md5 = next((f["md5"] for f in info["files"] if f["file"].endswith(name)), "")
        PROVENANCE.append({"Manuscript_Item": item, "Output": out_file, "Source": f"results/{exp}/{name}",
                           "Notebook": info["notebook"], "Run_Finished": info["finished"],
                           "Device": info["environment"]["device"], "Lib_Version": info["environment"]["rareguard_lib"],
                           "Source_MD5": md5})
    print(f"{item}: from " + ", ".join(f"results/{e}/{n}" for e, n in sources))

def save_fig(fig, name, item, sources):
    fig.savefig(os.path.join(FIG, name + ".pdf"))
    fig.savefig(os.path.join(FIG, name + ".png"), dpi=300)
    record(item, f"manuscript/figures/{name}.pdf", sources)

def save_tex(name, text, item, sources):
    head = f"% {item}: generated by 09_manuscript_figures_tables.ipynb on {datetime.date.today()}\n"
    head += "".join(f"% source: results/{e}/{n}\n" for e, n in sources)
    with open(os.path.join(TAB, name), "w") as fh:
        fh.write(head + text)
    record(item, f"manuscript/tables/{name}", sources)

TESTS = rg.read_result(cfg, "stats", "paired_tests.csv")
def test(family, condition, metric):
    r = TESTS[(TESTS.Family == family) & (TESTS.Condition.astype(str) == str(condition)) & (TESTS.Metric == metric)]
    assert len(r) == 1, (family, condition, metric)
    return r.iloc[0]

def pm(mean, sd, d=3):
    return f"${mean:.{d}f}\\pm{sd:.{d}f}$"

def fmt_p(p):
    return "$<$0.001" if p < 0.001 else (f"{p:.3f}" if p < 0.1 else f"{p:.2f}")

# Colour-blind-checked palette; series also differ by line style or marker
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK, GRID = "#333333", "#e3e2de"
plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.edgecolor": "#888", "axes.linewidth": 0.6,
                     "xtick.color": INK, "ytick.color": INK, "axes.labelcolor": INK, "legend.frameon": False,
                     "lines.linewidth": 1.6, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42})
WIDTH = 3.45   # IEEE single column, inches

def grid(ax, axis="both"):
    ax.grid(True, axis=axis, color=GRID, lw=0.5); ax.set_axisbelow(True)

## Figure 2: precision-recall curves, seed 42

In [ ]:
z = rg.load_scores(cfg, "E1_components", "ulb_seed42")
W = cfg["fusion_weights"]
series = [("RAREGuard", rg.weighted_fusion(rg.scores_from_npz(z, "test"), W)),
          ("OCSVM", z["OCSVM_test"]), ("IF", z["IF_test"]), ("AE", z["AE_test"])]
fig, ax = plt.subplots(figsize=(WIDTH, 2.5)); grid(ax)
for (label, s), c, ls in zip(series, COL, ["-", "--", "-.", ":"]):
    p, r, _ = precision_recall_curve(z["y_test"], s)
    ax.plot(r, p, color=c, ls=ls, label=f"{label} (AP = {average_precision_score(z['y_test'], s):.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.legend(loc="upper right", fontsize=7)
fig.tight_layout(); save_fig(fig, "fig_pr_seed42", "Figure 2", [("E1_components", "scores/ulb_seed42.npz")]); plt.show()

## Figure 3: fusion configurations

In [ ]:
NAMES = {"RAREGuard_30_30_40": "RAREGuard (0.3/0.3/0.4)", "Equal_3": "Equal weights (3)", "OCSVM_AE_50_50": "OCSVM + AE",
         "Median_3": "Median (3)", "IF_OCSVM_50_50": "IF + OCSVM", "OCSVM_Only": "OCSVM only", "Rank_Mean_3": "Rank mean (3)",
         "IF_AE_50_50": "IF + AE", "IF_Only": "IF only", "AE_Only": "AE only"}
f = rg.read_result(cfg, "E2_E3_fusion", "fusion_baselines_summary.csv").sort_values("PR_AUC_mean")
yy = np.arange(len(f))
fig, ax = plt.subplots(figsize=(WIDTH, 2.9)); grid(ax, "x")
ax.errorbar(f.F1_mean, yy + 0.15, xerr=f.F1_sd, fmt="s", color=COL[1], ms=4, capsize=2, lw=1, label="F1-score")
ax.errorbar(f.PR_AUC_mean, yy - 0.15, xerr=f.PR_AUC_sd, fmt="o", color=COL[0], ms=4, capsize=2, lw=1, label="PR-AUC")
ax.set_yticks(yy); ax.set_yticklabels([NAMES[c] for c in f.Configuration], fontsize=7)
ax.set_xlabel("Score (mean ± SD, 5 splits)"); ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=7)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
fig.tight_layout(); save_fig(fig, "fig_fusion", "Figure 3", [("E2_E3_fusion", "fusion_baselines_summary.csv")]); plt.show()

## Figure 4: assumed contamination c

In [ ]:
cs = rg.read_result(cfg, "E4_E5_thresholds", "assumed_contamination_summary.csv").sort_values("Contamination")
xc = np.arange(len(cs))
fig, ax = plt.subplots(figsize=(WIDTH, 2.3)); grid(ax)
for m, c, mk, ls, lab in [("Precision", COL[0], "o", "-", "Precision"), ("Recall", COL[1], "s", "--", "Recall"),
                          ("F1", COL[2], "^", "-.", "F1-score")]:
    ax.errorbar(xc, cs[f"{m}_mean"], yerr=cs[f"{m}_sd"], color=c, marker=mk, ms=4.5, capsize=2.5, ls=ls, label=lab)
prevalence = 100 * 492 / 284807
xs = 100 * cs.Contamination.to_numpy()
pos = np.interp(prevalence, xs, xc)   # place the prevalence line on the categorical axis
ax.axvline(pos, color="#999", lw=0.8, ls=":")
ax.text(pos + 0.06, 0.97, f"true prevalence\n({prevalence:.2f}%)", fontsize=6.5, color="#666", va="top")
ax.set_xlabel("Assumed contamination c (%)"); ax.set_ylabel("Score (mean ± SD, 5 splits)"); ax.set_ylim(0, 1)
ax.set_xticks(xc); ax.set_xticklabels([f"{v:.2f}" for v in xs]); ax.legend(loc="upper right", fontsize=7)
fig.tight_layout(); save_fig(fig, "fig_threshold_c", "Figure 4", [("E4_E5_thresholds", "assumed_contamination_summary.csv")]); plt.show()

## Figure 5: fraud in the training reference

In [ ]:
g = rg.read_result(cfg, "E6_reference_contamination", "reference_contamination_summary.csv").sort_values("Training_Reference_Contamination")
x = np.arange(len(g))
fig, ax = plt.subplots(figsize=(WIDTH, 2.4)); grid(ax)
for m, c, mk, lab in [("ROC_AUC", COL[0], "o", "ROC-AUC"), ("F1", COL[1], "s", "F1-score"), ("PR_AUC", COL[2], "^", "PR-AUC")]:
    ax.errorbar(x, g[f"{m}_mean"], yerr=g[f"{m}_sd"], color=c, marker=mk, ms=4.5, capsize=2.5, lw=1.6)
    ax.annotate(lab, (x[-1], g[f"{m}_mean"].iloc[-1]), xytext=(6, 0), textcoords="offset points", va="center", fontsize=7, color=INK)
ax.set_xlabel("Fraud in training reference (%)"); ax.set_ylabel("Score (mean ± SD, 5 splits)")
ax.set_ylim(0, 1); ax.set_xlim(-0.3, len(g) + 0.1); ax.set_xticks(x)
ax.set_xticklabels([f"{100 * v:g}" for v in g.Training_Reference_Contamination])
fig.tight_layout(); save_fig(fig, "fig_contamination", "Figure 5", [("E6_reference_contamination", "reference_contamination_summary.csv")]); plt.show()

## Table 1: component comparison

In [ ]:
s = rg.read_result(cfg, "E1_components", "summary_mean_sd.csv")
s = s[s.Threshold_Mode == "Fully unsupervised"].set_index("Experiment")
rows = [("IF_Only", "IF only"), ("OCSVM_Only", "OCSVM only"), ("AE_Only", "AE only"), ("RAREGuard_No_FS", "RAREGuard, no FS"),
        ("RAREGuard_No_PCA", "RAREGuard, no PCA"), ("RAREGuard_Full", "RAREGuard (complete)")]
metrics = ["Precision", "Recall", "F1", "PR_AUC", "ROC_AUC", "MCC"]
best = {m: s.loc[[k for k, _ in rows], f"{m}_mean"].idxmax() for m in metrics}
body = []
for key, lab in rows:
    cells = [f"$\\mathbf{{{s.loc[key, m + '_mean']:.3f}}}\\pm{s.loc[key, m + '_sd']:.3f}$" if best[m] == key
             else pm(s.loc[key, m + "_mean"], s.loc[key, m + "_sd"]) for m in metrics]
    body.append(f"{lab} & " + " & ".join(cells) + r" \\")
tex = r"""\begin{table*}[t]
\centering
\caption{Component comparison on ULB under fraud-label-free threshold calibration ($c=0.002$). Mean $\pm$ SD over five stratified splits; best mean in bold.}
\label{tab:main}
\small
\begin{tabular}{lcccccc}
\toprule
Configuration & Precision & Recall & F1-score & PR-AUC & ROC-AUC & MCC \\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table*}
"""
save_tex("tab_main.tex", tex, "Table 1", [("E1_components", "summary_mean_sd.csv")]); print(tex)

## Table 2: fusion baselines, paired differences with Holm correction

In [ ]:
order = ["Equal_3", "OCSVM_AE_50_50", "Median_3", "IF_OCSVM_50_50", "OCSVM_Only", "Rank_Mean_3", "IF_AE_50_50", "IF_Only", "AE_Only"]
body = []
for c in order:
    f1, pr = test("E2 fusion baselines", c, "F1"), test("E2 fusion baselines", c, "PR_AUC")
    body.append(f"{NAMES[c]} & ${f1.Delta:+.3f}$ $[{f1.CI95_Low:.3f}, {f1.CI95_High:.3f}]$ & {fmt_p(f1.p_Holm)} & "
                f"${pr.Delta:+.3f}$ $[{pr.CI95_Low:.3f}, {pr.CI95_High:.3f}]$ & {fmt_p(pr.p_Holm)} \\\\")
tex = r"""\begin{table*}[t]
\centering
\caption{Paired differences from the reference fusion (configuration minus RAREGuard) over five matched splits, with 95\% CIs and Holm-adjusted $p$-values ($p_{\mathrm{H}}$, nine comparisons per metric).}
\label{tab:fusion}
\small
\begin{tabular}{lcccc}
\toprule
Configuration & $\Delta$F1-score [95\% CI] & $p_{\mathrm{H}}$ & $\Delta$PR-AUC [95\% CI] & $p_{\mathrm{H}}$ \\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table*}
"""
save_tex("tab_fusion.tex", tex, "Table 2", [("E2_E3_fusion", "fusion_baselines_by_seed.csv"), ("stats", "paired_tests.csv")]); print(tex)

## Table 3: label-free threshold estimators

In [ ]:
t = rg.read_result(cfg, "E4_E5_thresholds", "threshold_methods_summary.csv").set_index("Method")
labels = [("Percentile", "Percentile"), ("EVT_GPD", "EVT (GPD)"), ("MAD", "MAD"), ("IQR", "IQR")]
body = [f"{lab} & " + " & ".join(pm(t.loc[k, m + "_mean"], t.loc[k, m + "_sd"]) for m in ["Precision", "Recall", "F1", "MCC"])
        + f" & {100 * t.loc[k, 'Test_Alert_Rate_mean']:.2f}\\% \\\\" for k, lab in labels]
for k, lab in labels[1:]:
    r = test("E4 threshold estimators", k, "F1")
    print(f"{lab:10s} vs percentile: dF1 = {r.Delta:+.3f} [{r.CI95_Low:.3f}, {r.CI95_High:.3f}], p_H = {fmt_p(r.p_Holm)}")
tex = r"""\begin{table*}[t]
\centering
\caption{Label-free threshold estimators on identical fused scores (mean $\pm$ SD over five splits).}
\label{tab:threshold}
\begin{tabular}{lccccc}
\toprule
Estimator & Precision & Recall & F1-score & MCC & Alerts \\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table*}
"""
save_tex("tab_threshold.tex", tex, "Table 3", [("E4_E5_thresholds", "threshold_methods_summary.csv"), ("stats", "paired_tests.csv")]); print(tex)

## Table 4: training-reference contamination

In [ ]:
g = rg.read_result(cfg, "E6_reference_contamination", "reference_contamination_summary.csv").set_index("Training_Reference_Contamination")
levels = sorted(g.index)
body = []
for l in levels:
    cells = " & ".join(pm(g.loc[l, m + "_mean"], g.loc[l, m + "_sd"]) for m in ["F1", "PR_AUC", "ROC_AUC"])
    if l == levels[0]:
        body.append(f"0\\% & {cells} & -- & -- \\\\")
    else:
        r = test("E6 reference contamination", l, "PR_AUC")
        body.append(f"{100 * l:.1f}\\% & {cells} & ${r.Delta:.3f}$ $[{r.CI95_Low:.3f},{r.CI95_High:.3f}]$ & {fmt_p(r.p_Holm)} \\\\")
tex = r"""\begin{table*}[t]
\centering
\caption{RAREGuard with a 30{,}000-observation training reference containing increasing fraud. Mean $\pm$ SD over five splits; $\Delta$PR-AUC is the paired difference from the clean reference, with Holm-adjusted $p$-values over the four levels.}
\label{tab:contam}
\begin{tabular}{lccccc}
\toprule
Fraud & F1-score & PR-AUC & ROC-AUC & $\Delta$PR-AUC [95\% CI] & $p_{\mathrm{H}}$ \\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table*}
"""
save_tex("tab_contam.tex", tex, "Table 4", [("E6_reference_contamination", "reference_contamination_summary.csv"), ("stats", "paired_tests.csv")]); print(tex)

## Table 5: run time

In [ ]:
cost = rg.read_result(cfg, "E8_cost", "cost_by_configuration.csv").set_index("Experiment")
rows = [("OCSVM_Only", "OCSVM only"), ("IF_Only", "IF only"), ("AE_Only", "AE only"), ("RAREGuard_Full", "RAREGuard")]
body = [f"{lab} & {pm(cost.loc[k, 'Train_Time_s_mean'], cost.loc[k, 'Train_Time_s_sd'], 2)} & "
        f"{pm(cost.loc[k, 'Total_Time_s_mean'], cost.loc[k, 'Total_Time_s_sd'], 2)} \\\\" for k, lab in rows]
print("Device for these timings:", run_info("E1_components")["environment"]["device"])
tex = r"""\begin{table}[t]
\centering
\caption{Run time on ULB (seconds, mean $\pm$ SD over five splits).}
\label{tab:cost}
\begin{tabular}{lcc}
\toprule
Configuration & Training & Total \\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table}
"""
save_tex("tab_cost.tex", tex, "Table 5", [("E8_cost", "cost_by_configuration.csv")]); print(tex)

## Table 6: PaySim

In [ ]:
ps = rg.read_result(cfg, "E9_paysim", "representation_summary.csv").set_index("Representation")
reps = rg.read_result(cfg, "E9_paysim", "representation_by_seed.csv")
n_base = int(reps.loc[reps.Representation == "Baseline", "Feature_Count"].iloc[0])
n_enh = int(reps.loc[reps.Representation == "Enhanced", "Feature_Count"].iloc[0])
lab = {"Precision": "Precision", "Recall": "Recall", "F1": "F1-score", "MCC": "MCC", "PR_AUC": "PR-AUC", "ROC_AUC": "ROC-AUC"}
body = [f"{lab[m]} & {pm(ps.loc['Baseline', m + '_mean'], ps.loc['Baseline', m + '_sd'])} & "
        f"{pm(ps.loc['Enhanced', m + '_mean'], ps.loc['Enhanced', m + '_sd'])} \\\\" for m in lab]
r = test("E9 PaySim representation", "Enhanced", "ROC_AUC")
print(f"Enhanced - Baseline ROC-AUC: {r.Delta:+.3f} [{r.CI95_Low:.3f}, {r.CI95_High:.3f}]")
tex = r"""\begin{table}[t]
\centering
\caption{RAREGuard on a 300{,}000-transaction PaySim sample (mean $\pm$ SD over five splits). The enhanced representation is exploratory.}
\label{tab:paysim}
\small
\begin{tabular}{lcc}
\toprule
""" + f"Metric & Baseline ({n_base} features) & Enhanced ({n_enh} features) \\\\\n" + r"""\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table}
"""
save_tex("tab_paysim.tex", tex, "Table 6", [("E9_paysim", "representation_summary.csv"), ("stats", "paired_tests.csv")]); print(tex)

## Provenance file and download

In [ ]:
prov = pd.DataFrame(PROVENANCE)
prov.to_csv(rg.proj(cfg, "manuscript", "provenance.csv"), index=False)
display(prov)

zip_path = rg.proj(cfg, "manuscript", "rareguard_figures_tables.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    for folder, arc in [(FIG, "figures"), (TAB, "tables")]:
        for fn in sorted(os.listdir(folder)):
            if fn.endswith((".pdf", ".tex")):
                zf.write(os.path.join(folder, fn), os.path.join(arc, fn))
    zf.write(rg.proj(cfg, "manuscript", "provenance.csv"), "provenance.csv")
print("Saved:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass